# 06 — Predictive Modeling
## Tamil Nadu Assembly Elections

**Objectives:**
- Feature engineering from historical data
- Win/loss classifier (Random Forest / XGBoost)
- Margin regression
- SHAP feature importance
- 2026 scenario modeling

In [ ]:
import sys
sys.path.insert(0, '/app')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import shap

from db import query, query_all, GENERAL_ELECTION_YEARS, POST_DELIM_YEARS, MAJOR_PARTIES, ALLIANCES

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

df = query_all()
ge = df[df.year.isin(GENERAL_ELECTION_YEARS)].copy()
print(f'General election records: {len(ge):,}')

## 1. Feature Engineering

In [ ]:
# Build features for post-delimitation elections (2011, 2016, 2021)
# For each candidate-constituency pair, create features from current and historical data

def get_previous_election(year):
    """Get the previous general election year."""
    idx = GENERAL_ELECTION_YEARS.index(year)
    return GENERAL_ELECTION_YEARS[idx - 1] if idx > 0 else None


def build_features(ge_df, target_year):
    """Build feature matrix for a given election year."""
    prev_year = get_previous_election(target_year)
    if prev_year is None:
        return pd.DataFrame()
    
    curr = ge_df[ge_df.year == target_year].copy()
    prev = ge_df[ge_df.year == prev_year].copy()
    
    # Previous election winners by constituency
    prev_winners = prev[prev.position == 1][['constituency_no', 'party', 'margin_percentage', 
                                              'vote_share_percentage', 'turnout_percentage']].copy()
    prev_winners.columns = ['constituency_no', 'prev_winner_party', 'prev_margin_pct', 
                             'prev_winner_vote_share', 'prev_turnout']
    
    # Party strength in previous election (state-level)
    prev_party_strength = prev.groupby('party').agg(
        prev_party_seats=('position', lambda x: (x == 1).sum()),
        prev_party_avg_vote_share=('vote_share_percentage', 'mean')
    ).reset_index()
    
    features = curr.merge(prev_winners, on='constituency_no', how='left')
    features = features.merge(prev_party_strength, on='party', how='left')
    
    # Derived features
    features['is_incumbent_party'] = (features.party == features.prev_winner_party).astype(int)
    features['is_incumbent'] = (features.incumbent == 'TRUE').astype(int)
    features['is_turncoat'] = (features.turncoat == 'TRUE').astype(int)
    features['is_recontest'] = (features.recontest == 'TRUE').astype(int)
    features['is_female'] = (features.sex == 'F').astype(int)
    features['is_same_constituency'] = (features.same_constituency == 'TRUE').astype(int)
    features['is_major_party'] = features.party.isin(MAJOR_PARTIES).astype(int)
    features['turnout_change'] = features.turnout_percentage - features.prev_turnout
    
    # Target
    features['is_winner'] = (features.position == 1).astype(int)
    
    return features


# Build features for 2011, 2016, 2021
all_features = []
for year in POST_DELIM_YEARS:
    feat = build_features(ge, year)
    if len(feat) > 0:
        all_features.append(feat)
        print(f'{year}: {len(feat)} records, {feat.is_winner.sum()} winners')

features_df = pd.concat(all_features, ignore_index=True)
print(f'\nTotal feature records: {len(features_df):,}')

In [ ]:
# Select model features
feature_cols = [
    'age', 'n_cand', 'enop', 'turnout_percentage',
    'prev_margin_pct', 'prev_winner_vote_share', 'prev_turnout',
    'prev_party_seats', 'prev_party_avg_vote_share',
    'is_incumbent_party', 'is_incumbent', 'is_turncoat', 'is_recontest',
    'is_female', 'is_same_constituency', 'is_major_party',
    'turnout_change', 'no_terms', 'contested'
]

X = features_df[feature_cols].copy()
y = features_df['is_winner'].copy()

# Handle missing values
X = X.fillna(0)

print(f'Feature matrix: {X.shape}')
print(f'Target balance: {y.value_counts().to_dict()}')
print(f'Win rate: {y.mean()*100:.1f}%')

## 2. Win/Loss Classifier

In [ ]:
# Train/test split: train on 2011+2016, test on 2021
train_mask = features_df.year.isin([2011, 2016])
test_mask = features_df.year == 2021

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f'Train: {len(X_train)} ({y_train.sum()} winners)')
print(f'Test:  {len(X_test)} ({y_test.sum()} winners)')

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=10,
                            class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest Results (Test: 2021) ===')
print(f'Accuracy: {accuracy_score(y_test, rf_pred):.3f}')
print(f'AUC-ROC: {roc_auc_score(y_test, rf_proba):.3f}')
print()
print(classification_report(y_test, rf_pred, target_names=['Loser', 'Winner']))

In [ ]:
# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum(),
    eval_metric='logloss', random_state=42, n_jobs=-1
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print('=== XGBoost Results (Test: 2021) ===')
print(f'Accuracy: {accuracy_score(y_test, xgb_pred):.3f}')
print(f'AUC-ROC: {roc_auc_score(y_test, xgb_proba):.3f}')
print()
print(classification_report(y_test, xgb_pred, target_names=['Loser', 'Winner']))

In [ ]:
# Confusion matrices side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, title in [(ax1, rf_pred, 'Random Forest'), (ax2, xgb_pred, 'XGBoost')]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Loser', 'Winner'], yticklabels=['Loser', 'Winner'])
    ax.set_title(f'{title} — Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('/app/output/06_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Importance (SHAP)

In [ ]:
# SHAP analysis on XGBoost model
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False, max_display=15)
plt.title('SHAP Feature Importance — XGBoost Win/Loss Model', fontsize=14)
plt.tight_layout()
plt.savefig('/app/output/06_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance comparison: RF vs XGBoost
rf_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
xgb_importance = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
rf_importance.plot(kind='barh', ax=ax1, color='#2196F3', edgecolor='white')
ax1.set_title('Random Forest — Feature Importance')

xgb_importance.plot(kind='barh', ax=ax2, color='#FF7043', edgecolor='white')
ax2.set_title('XGBoost — Feature Importance')

plt.tight_layout()
plt.savefig('/app/output/06_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Margin Regression

In [ ]:
# Predict margin percentage for winners only
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

winners_feat = features_df[features_df.is_winner == 1].copy()
X_reg = winners_feat[feature_cols].fillna(0)
y_reg = winners_feat['margin_percentage'].fillna(0)

train_mask_reg = winners_feat.year.isin([2011, 2016])
test_mask_reg = winners_feat.year == 2021

X_train_reg, X_test_reg = X_reg[train_mask_reg], X_reg[test_mask_reg]
y_train_reg, y_test_reg = y_reg[train_mask_reg], y_reg[test_mask_reg]

gbr = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
gbr.fit(X_train_reg, y_train_reg)
margin_pred = gbr.predict(X_test_reg)

print('=== Margin Regression (Winners Only, Test: 2021) ===')
print(f'R² Score: {r2_score(y_test_reg, margin_pred):.3f}')
print(f'MAE: {mean_absolute_error(y_test_reg, margin_pred):.2f}%')

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test_reg, margin_pred, alpha=0.5, s=30, color='#2196F3')
max_val = max(y_test_reg.max(), margin_pred.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1, label='Perfect prediction')
ax.set_title('Predicted vs Actual Winner Margin (2021)')
ax.set_xlabel('Actual Margin (%)')
ax.set_ylabel('Predicted Margin (%)')
ax.legend()
plt.tight_layout()
plt.savefig('/app/output/06_margin_regression.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 2026 Scenario Modeling

In [ ]:
# Use 2021 data to project 2026 scenarios
# Key assumption: anti-incumbency factor (historical avg)

data_2021 = ge[ge.year == 2021].copy()
winners_2021 = data_2021[data_2021.position == 1].copy()

# Historical anti-incumbency: ruling party loses avg X% seat share
# Calculate from model features
print('=== 2021 Baseline ===')
print(f'DMK seats: {(winners_2021.party == "DMK").sum()}')
print(f'ADMK seats: {(winners_2021.party == "ADMK").sum()}')
print(f'Total seats: {len(winners_2021)}')
print()

# Using XGBoost model — simulate 2026 by flipping incumbency
# For each 2021 constituency, the winner becomes the incumbent in 2026
# Create hypothetical feature set
scenario_base = features_df[features_df.year == 2021].copy()

# Scenario 1: Strong anti-incumbency (like 2011, 2021)
# Scenario 2: Weak anti-incumbency (like 2016)
# Score each constituency by model's predicted probability

scenario_base['win_probability'] = xgb_model.predict_proba(X_test)[:, 1]

# Top vulnerable DMK seats (DMK won in 2021 with lowest predicted probability)
dmk_vulnerable = scenario_base[(scenario_base.party == 'DMK') & (scenario_base.is_winner == 1)]
dmk_vulnerable = dmk_vulnerable.nsmallest(20, 'win_probability')[['constituency_name', 'candidate', 'margin_percentage', 'win_probability']]

print('=== 20 Most Vulnerable DMK Seats (2021 Winners) ===')
print(dmk_vulnerable.to_string(index=False))

# ADMK seats with highest upset potential
admk_strong = scenario_base[(scenario_base.party == 'ADMK') & (scenario_base.is_winner == 1)]
admk_strong = admk_strong.nlargest(15, 'win_probability')[['constituency_name', 'candidate', 'margin_percentage', 'win_probability']]

print('\n=== 15 Safest ADMK Seats (2021 Winners) ===')
print(admk_strong.to_string(index=False))

In [ ]:
# Model summary
print('=== PREDICTIVE MODEL SUMMARY ===')
print(f'\nClassification (XGBoost):')
print(f'  Train years: 2011, 2016 | Test year: 2021')
print(f'  Accuracy: {accuracy_score(y_test, xgb_pred):.3f}')
print(f'  AUC-ROC: {roc_auc_score(y_test, xgb_proba):.3f}')
print(f'\nMargin Regression (GBR):')
print(f'  R² Score: {r2_score(y_test_reg, margin_pred):.3f}')
print(f'  MAE: {mean_absolute_error(y_test_reg, margin_pred):.2f}%')
print(f'\nTop 3 features (XGBoost):')
top3 = xgb_importance.tail(3)
for feat, imp in zip(top3.index[::-1], top3.values[::-1]):
    print(f'  {feat}: {imp:.4f}')